# CSCI 5931 — Exam 3 Practice Problems (Runnable Companion)

**Use with:** `07_Practice_Problems.md`

This notebook lets you **run** the answers to exam-style problems and verify your hand calculations. Every code cell here is exam-ready: the patterns match what Dr. B would test (parameter counting, shape arithmetic, LSTM forward step, attention scoring, BPTT chain).

**How to use:**
1. Read each problem in the markdown cell
2. Try solving it on paper FIRST
3. Run the code cell to verify
4. If you got it wrong, study the code's logic — that pattern shows up on the exam

## Setup

In [ ]:
import numpy as np
import math
np.random.seed(42)

# torch is optional — needed only for shape-verification cells in Section 2 + 5
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    torch.manual_seed(42)
    HAS_TORCH = True
    print(f'numpy {np.__version__} | torch {torch.__version__}  ✅ ready')
except ImportError:
    HAS_TORCH = False
    print(f'numpy {np.__version__}  ⚠️  torch not installed — install with: pip install torch')
    print('  Sections 1, 3, 4 work in pure numpy. Sections 2, 5 require torch.')

---
# Section 1: Parameter Counting Drills

Memorize these formulas:

| Layer | Params (with bias) |
|---|---|
| `nn.Linear(in, out)` | $(\text{in} + 1) \cdot \text{out}$ |
| `nn.Conv2d(c_in, c_out, k)` | $(k^2 \cdot c_\text{in} + 1) \cdot c_\text{out}$ |
| Vanilla RNN (no biases) | $2HC + H^2$ |
| LSTM gate matrix `Wx` | $4H \cdot (H+D)$ |
| Multi-head attention | $4 d_\text{model}^2$ |


### Problem 1.1 — Vanilla RNN params (with biases)
Vocab $C = 3000$, hidden $H = 128$. Total trainable parameters?

In [ ]:
C, H = 3000, 128
U_params = H * C
W_params = H * H
V_params = C * H
bh_params = H
by_params = C
total = U_params + W_params + V_params + bh_params + by_params

print(f'U: {U_params:>10,}  ({H}x{C})')
print(f'W: {W_params:>10,}  ({H}x{H})')
print(f'V: {V_params:>10,}  ({C}x{H})')
print(f'b_h: {bh_params:>8,}')
print(f'b_y: {by_params:>8,}')
print(f'-------------------')
print(f'TOTAL: {total:,}')
print()
print(f'Quick formula 2HC+H² (no biases): {2*H*C + H*H:,}')

### Problem 1.2 — LSTM gate parameters
Hidden $H=64$, input $D=100$ (HW-4 style: combined gate matrix). Compute `Wx` and `b` params.

In [ ]:
H, D = 64, 100
Wx_shape = (4*H, H+D)
Wx_params = Wx_shape[0] * Wx_shape[1]
b_params = 4 * H
V_params = D * H
by_params = D

print(f'Combined gate matrix Wx shape: {Wx_shape}')
print(f'  Wx params: {Wx_params:,}')
print(f'  b  params: {b_params:,}')
print(f'  Output V (D x H): {V_params:,}')
print(f'  Output b_y: {by_params:,}')
print(f'  TOTAL: {Wx_params + b_params + V_params + by_params:,}')
print()
print(f'Formula 4H(H+D)+4H+DH+D = {4*H*(H+D) + 4*H + D*H + D:,}')

### Problem 1.3 — SimpleGAN Discriminator
Architecture: 784 → 512 → 256 → 1, all with biases. Total params?

In [ ]:
layers = [(784, 512), (512, 256), (256, 1)]
total = 0
for i, (in_f, out_f) in enumerate(layers, 1):
    p = (in_f + 1) * out_f
    total += p
    print(f'  Layer {i}: ({in_f}+1) * {out_f} = {p:>9,}')
print(f'  ----------')
print(f'  TOTAL:    {total:,}')

if HAS_TORCH:
    D = nn.Sequential(
        nn.Linear(784, 512), nn.LeakyReLU(0.2),
        nn.Linear(512, 256), nn.LeakyReLU(0.2),
        nn.Linear(256, 1)
    )
    pytorch_count = sum(p.numel() for p in D.parameters())
    print(f'\n  PyTorch verification: {pytorch_count:,} ✅' if pytorch_count == total else f'\n  Mismatch: {pytorch_count}')

### Problem 1.4 — Multi-Head Attention
$d_\text{model} = 1024$, $h = 16$ heads. Per-head $d_k$? MHA params (no biases)?

In [ ]:
d_model = 1024
h = 16
d_k = d_model // h
mha_params = 4 * d_model * d_model

print(f'Per-head dim d_k = {d_model}/{h} = {d_k}')
print(f'MHA params 4·d² = 4·{d_model}² = {mha_params:,}  ({mha_params/1e6:.2f}M)')
print()
print('The 4·d² breaks down as:')
print(f'  W_Q: d_model × d_model = {d_model}² = {d_model**2:,}')
print(f'  W_K: same = {d_model**2:,}')
print(f'  W_V: same = {d_model**2:,}')
print(f'  W_O: same = {d_model**2:,}')
print(f'  Total: {4*d_model**2:,}')

### Problem 1.5 — U-Net DoubleConv
`DoubleConv(in=256, out=512)` — two 3×3 convs with biases. Params?

In [ ]:
in_ch, out_ch = 256, 512
conv1 = (3 * 3 * in_ch + 1) * out_ch
conv2 = (3 * 3 * out_ch + 1) * out_ch
total = conv1 + conv2

print(f'Conv1: (3·3·{in_ch}+1)·{out_ch} = {conv1:>10,}')
print(f'Conv2: (3·3·{out_ch}+1)·{out_ch} = {conv2:>10,}')
print(f'TOTAL:                           {total:,}  ({total/1e6:.2f}M)')

if HAS_TORCH:
    block = nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1),
        nn.ReLU(),
        nn.Conv2d(out_ch, out_ch, 3, padding=1),
        nn.ReLU()
    )
    pytorch_count = sum(p.numel() for p in block.parameters())
    print(f'\nPyTorch verification: {pytorch_count:,} ✅' if pytorch_count == total else f'\nMismatch')

---
# Section 2: Shape Arithmetic (verified by PyTorch)

**Conv shape:** $\lfloor (N + 2p - f)/s \rfloor + 1$

**Transposed conv shape:** $(N - 1) \cdot s - 2p + f$  ← DIFFERENT formula!

### Problem 2.1 — Conv output
Input $32 \times 32 \times 3$, kernel $5 \times 5$, padding 0, stride 1, 16 filters. Output?

In [ ]:
N_in, p, f, s, k = 32, 0, 5, 1, 16
N_out = (N_in + 2*p - f) // s + 1
print(f'Hand-calc: ⌊({N_in}+{2*p}-{f})/{s}⌋+1 = {N_out}')
print(f'Output: {N_out}×{N_out}×{k}')

if HAS_TORCH:
    x = torch.zeros(1, 3, N_in, N_in)
    conv = nn.Conv2d(3, k, kernel_size=f, padding=p, stride=s)
    out = conv(x)
    print(f'PyTorch: {tuple(out.shape)} → spatial {out.shape[-1]}×{out.shape[-2]}, channels {out.shape[1]} ✅')

### Problem 2.3 — Transposed conv
Input $8 \times 8$, kernel $4 \times 4$, padding $p=1$, stride $s=2$. Output?

In [ ]:
N_in, p, f, s = 8, 1, 4, 2
N_out = (N_in - 1) * s - 2*p + f
print(f'Hand-calc: ({N_in}-1)·{s} - 2·{p} + {f} = {N_out}')

if HAS_TORCH:
    x = torch.zeros(1, 8, N_in, N_in)
    upconv = nn.ConvTranspose2d(8, 4, kernel_size=f, padding=p, stride=s)
    out = upconv(x)
    print(f'PyTorch: {tuple(out.shape)} → output spatial {out.shape[-1]}×{out.shape[-2]} ✅')

### Problem 2.4 — U-Net concat after upsample
`c4` is $(1, 256, 16, 16)$. Apply `ConvTranspose2d(256, 128, k=2, s=2)` and concat with skip $(1, 128, 32, 32)$. Resulting shape?

In [ ]:
if HAS_TORCH:
    c4 = torch.zeros(1, 256, 16, 16)
    skip = torch.zeros(1, 128, 32, 32)
    upconv = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
    upsampled = upconv(c4)
    print(f'After upconv: {tuple(upsampled.shape)}')
    cat = torch.cat([upsampled, skip], dim=1)
    print(f'After concat: {tuple(cat.shape)}  ← channels = {upsampled.shape[1]} + {skip.shape[1]}')
else:
    print('Hand-calc:')
    print('  upconv((16-1)·2 - 0 + 2 = 32 spatial), 128 channels → (1, 128, 32, 32)')
    print('  concat with skip(128 ch) along channel dim → (1, 256, 32, 32)')

### Problem 2.5 — Self-attention shapes
$Q, K, V$ all of shape $(50, 64)$. Score matrix? Output?

In [ ]:
n, d_k = 50, 64
Q = np.random.randn(n, d_k)
K = np.random.randn(n, d_k)
V = np.random.randn(n, d_k)

scores = Q @ K.T / np.sqrt(d_k)
weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)  # softmax
output = weights @ V

print(f'Q.shape:      {Q.shape}')
print(f'K.shape:      {K.shape}')
print(f'V.shape:      {V.shape}')
print(f'Q@K.T:        {(Q@K.T).shape}    ← (n_q, n_k) score matrix')
print(f'After softmax:{weights.shape}    ← rows sum to 1.0')
print(f'@ V:          {output.shape}    ← same shape as input V')
print(f'Row sum check: weights[0].sum() = {weights[0].sum():.6f}')

---
# Section 3: Reverse Shape Derivation (Exam 2 Q6 style)

Given input/output shapes, find kernel size, stride, padding, or filter count.

### Problem 3.2 — Find $k$ and $n$
Input: $256 \times 256 \times 192$. Output: $256 \times 256 \times 32$. Stride 1, padding 0. Find kernel size $k$ and filter count $n$.

In [ ]:
N_in_spatial, N_out_spatial = 256, 256
channels_in, channels_out = 192, 32
p, s = 0, 1

# For spatial size unchanged with p=0, s=1:
# N_out = (N_in - k + 1)
# So 256 = 256 - k + 1 → k = 1
k = N_in_spatial - N_out_spatial + 1
n = channels_out

print(f'For spatial unchanged with p=0, s=1: k = N_in - N_out + 1 = {k}')
print(f'For output channels {channels_out}: n = {n} filters')
print(f'\nThis is a 1×1 convolution (channel-mixing only) with 32 filters.')

if HAS_TORCH:
    x = torch.zeros(1, channels_in, N_in_spatial, N_in_spatial)
    conv = nn.Conv2d(channels_in, n, kernel_size=k, padding=p, stride=s)
    out = conv(x)
    print(f'\nPyTorch verification: {tuple(out.shape)} ✅')

---
# Section 4: Computation Chains (numpy)

These demonstrate the actual forward / backward steps from HW-4.

### Problem 4.1 — Vanilla RNN forward step
$U \cdot x_t + W \cdot s_{t-1} + b_h$, then tanh.

Given $U=[0.3, 0.4]^T$, $W=[0.5, -0.2]^T$, $b_h=0.1$, $s_{t-1}=0$, $x_t=1.0$ (with $H=1$, $D=2$):

In [ ]:
# Toy 1-d hidden, 2-d input
U = np.array([[0.3, 0.4]])     # shape (H=1, D=2)
W = np.array([[0.5]])           # shape (H=1, H=1)
bh = np.array([[0.1]])
x_t = np.array([[1.0], [0.0]])  # shape (D=2, 1)
s_prev = np.array([[0.0]])

preact = U @ x_t + W @ s_prev + bh
s_t = np.tanh(preact)

print(f'Preactivation: U·x + W·s + b = {preact.flatten()[0]:.4f}')
print(f'tanh(preact) = s_t = {s_t.flatten()[0]:.4f}')

### Problem 4.2 — LSTM forward step (numeric example from C2-LSTM-intuition notebook)
$x_t = 0.5$, $h_{t-1} = 0.1$, $C_{t-1} = 0.2$. All weights = 1.0, all biases = 0.1.
Compute $f_t, i_t, \tilde{C}_t, C_t, o_t, h_t$.

In [ ]:
def sigmoid(x): return 1 / (1 + np.exp(-x))

x_t = 0.5
h_prev = 0.1
C_prev = 0.2
# Each gate's weight vector is [w_h, w_x] = [1.0, 1.0], with the bias listed:
Wf, bf = (1.0, 1.0), 0.1   # forget
Wi, bi = (1.0, 1.0), 0.05  # input
WC, bC = (1.0, 1.0), 0.0   # candidate (note: bias 0)
Wo, bo = (1.0, 1.0), 0.1   # output

f_t = sigmoid(Wf[0]*h_prev + Wf[1]*x_t + bf)
i_t = sigmoid(Wi[0]*h_prev + Wi[1]*x_t + bi)
C_tilde = np.tanh(WC[0]*h_prev + WC[1]*x_t + bC)
C_t = f_t * C_prev + i_t * C_tilde
o_t = sigmoid(Wo[0]*h_prev + Wo[1]*x_t + bo)
h_t = o_t * np.tanh(C_t)

print(f'f_t (forget):    σ(0.6+0.1)  = {f_t:.4f}    ← keep ~{f_t*100:.0f}% of old cell')
print(f'i_t (input):     σ(0.6+0.05) = {i_t:.4f}')
print(f'C̃_t (cand):     tanh(0.6)   = {C_tilde:.4f}')
print(f'C_t (cell):      {f_t:.3f}·{C_prev} + {i_t:.3f}·{C_tilde:.3f} = {C_t:.4f}')
print(f'o_t (output):    σ(0.6+0.1)  = {o_t:.4f}')
print(f'h_t (hidden):    {o_t:.3f}·tanh({C_t:.3f}) = {h_t:.4f}')
print()
print('Note: Worked example in C2-LSTM-intuition notebook gets C_t≈0.342, h_t≈0.214')
print('(slight differences due to weight matrix ordering — both forms are valid)')

### Problem 4.3 — Attention scoring
$q = [1, 0, 0, 1]$, three keys: $k_1=[1,0,0,0]$, $k_2=[0,1,0,0]$, $k_3=[1,0,0,1]$. Use $d_k=4$ scaling.

In [ ]:
q = np.array([1, 0, 0, 1])
K = np.array([
    [1, 0, 0, 0],   # k_1
    [0, 1, 0, 0],   # k_2
    [1, 0, 0, 1],   # k_3 — exact match for q
])
d_k = 4

raw_scores = q @ K.T
scaled = raw_scores / np.sqrt(d_k)
weights = np.exp(scaled) / np.exp(scaled).sum()

print(f'Raw scores q·k_i:   {raw_scores}')
print(f'Scaled by √{d_k}={np.sqrt(d_k):.1f}: {np.round(scaled, 3)}')
print(f'After softmax:      {np.round(weights, 3)}')
print(f'Sum check: {weights.sum():.6f}')
print()
print(f'Most attention to k_3 ({weights[2]*100:.1f}%) — it matches q exactly')

### Problem 4.4 — BPTT chain length empirical demo
Show vanishing gradient: each factor ≈ 0.5, chain over 20 steps.

In [ ]:
factor = 0.5  # typical magnitude of ∂s_j/∂s_{j-1}
steps = list(range(1, 21))
products = [factor**n for n in steps]

print(f'{"Steps":>6} | {"Product":>12} | {"% of original":>15}')
print('-' * 40)
for n, p in zip(steps[::2], products[::2]):  # every other step
    print(f'{n:>6} | {p:>12.6f} | {p*100:>14.4f}%')
print()
print(f'After {steps[-1]} steps: gradient is {products[-1]*100:.6f}% of original.')
print('This is why vanilla RNN cannot learn long-range dependencies.')
print('LSTM solves this via cell-state path with f_t ≈ 1 (no decay).')

---
# Section 5: Code Trace Drills (PyTorch — verify shapes by running)

### Problem 5.1 — SimpleGAN Discriminator output shape
Build the discriminator from `17_SimpleGAN_PyTorch_notebook.md`. For input `(32, 1, 28, 28)`, what's the output shape?

In [ ]:
if HAS_TORCH:
    class Discriminator(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(784, 128)
            self.fc2 = nn.Linear(128, 64)
            self.fc3 = nn.Linear(64, 32)
            self.fc4 = nn.Linear(32, 1)
        def forward(self, x):
            x = x.view(x.shape[0], -1)            # (32, 784)
            x = F.leaky_relu(self.fc1(x), 0.2)
            x = F.leaky_relu(self.fc2(x), 0.2)
            x = F.leaky_relu(self.fc3(x), 0.2)
            return self.fc4(x)                     # (32, 1) — single logit per image

    D = Discriminator()
    inp = torch.zeros(32, 1, 28, 28)
    out = D(inp)
    print(f'Input shape:  {tuple(inp.shape)}')
    print(f'Output shape: {tuple(out.shape)}  ← single logit per image (no sigmoid)')
    print(f'Total params: {sum(p.numel() for p in D.parameters()):,}')

### Problem 5.2 — Attention step function
`q.shape=(1,64)`, `K.shape=(10,64)`, `V.shape=(10,32)`. Output shape?

In [ ]:
def attention_step(q, K, V):
    d_k = q.shape[-1]
    scores = q @ K.T / math.sqrt(d_k)
    weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)
    return weights @ V

q = np.random.randn(1, 64)
K = np.random.randn(10, 64)
V = np.random.randn(10, 32)
out = attention_step(q, K, V)

print(f'q.shape: {q.shape}')
print(f'K.shape: {K.shape}')
print(f'V.shape: {V.shape}')
print(f'attention scores: ({q.shape[0]},{K.shape[0]}) = (1,10)')
print(f'output: {out.shape}  ← (1, d_v=32)')

### Problem 5.4 — Embedding lookup
`nn.Embedding(num_embeddings=10000, embedding_dim=300)`. Input shape `(1, 4)`. Output shape and param count?

In [ ]:
if HAS_TORCH:
    embedding = nn.Embedding(num_embeddings=10000, embedding_dim=300)
    inputs = torch.tensor([[1, 5, 100, 999]])
    emb = embedding(inputs)
    print(f'Input shape:  {tuple(inputs.shape)}  (token IDs)')
    print(f'Output shape: {tuple(emb.shape)}  (each token → 300-dim vector)')
    print(f'Embedding parameters: {sum(p.numel() for p in embedding.parameters()):,}')
    print(f'Formula: V·d = 10000·300 = {10000*300:,}')

### Problem 5.5 — Gradient check verification
Numerical (central difference) vs analytical gradient. Pass if relative error < 1e-3.

In [ ]:
def gradient_check(num_grad, ana_grad):
    rel_err = abs(num_grad - ana_grad) / (abs(num_grad) + abs(ana_grad) + 1e-12)
    status = 'PASSED ✅' if rel_err < 1e-3 else 'FAILED ❌'
    return rel_err, status

# From practice problem 5.5: num_grad=0.0832, dW=0.0831
rel_err, status = gradient_check(0.0832, 0.0831)
print(f'numerical = 0.0832, analytical = 0.0831')
print(f'rel_err = {rel_err:.6f}  → {status}')
print(f'(threshold = 1e-3 = 0.001)')
print()
# Test with a clearly broken backprop
rel_err, status = gradient_check(0.0832, 0.5000)
print(f'numerical = 0.0832, analytical = 0.5000 (bad!)')
print(f'rel_err = {rel_err:.6f}  → {status}')

---
# Section 6: GAN Training Loop Pattern
Demonstrates the critical patterns Dr. B emphasized (sum of losses, target=1 for G).

In [ ]:
if HAS_TORCH:
    # Mini SimpleGAN to demonstrate training pattern
    G = nn.Sequential(nn.Linear(100, 32), nn.LeakyReLU(0.2), nn.Linear(32, 784), nn.Tanh())
    D = nn.Sequential(nn.Linear(784, 128), nn.LeakyReLU(0.2), nn.Linear(128, 1))
    bce = nn.BCEWithLogitsLoss()
    
    batch_size = 16
    real = torch.randn(batch_size, 784)        # fake "real" data
    z = torch.randn(batch_size, 100)
    
    # ===== D-step =====
    real_labels = torch.ones(batch_size, 1)
    fake_labels = torch.zeros(batch_size, 1)
    
    d_real_logits = D(real)
    d_real_loss = bce(d_real_logits, real_labels)
    
    fake = G(z).detach()                       # ← detach: don't update G during D-step
    d_fake_logits = D(fake)
    d_fake_loss = bce(d_fake_logits, fake_labels)
    
    d_loss = d_real_loss + d_fake_loss          # ← SUM (Dr. B's emphasis)
    
    # ===== G-step =====
    z2 = torch.randn(batch_size, 100)
    fake = G(z2)                                # ← NOT detached: gradient flows into G
    g_logits = D(fake)
    g_loss = bce(g_logits, real_labels)         # ← target=1 (G wants D to think fakes are real)
    
    print(f'D real loss:  {d_real_loss.item():.4f}')
    print(f'D fake loss:  {d_fake_loss.item():.4f}')
    print(f'D total loss: {d_loss.item():.4f}  ← REAL + FAKE (sum)')
    print(f'G loss:       {g_loss.item():.4f}  ← uses target=1 (real labels) on FAKE images')
    print()
    print('Key patterns to memorize:')
    print(' 1. d_loss = d_real_loss + d_fake_loss (SUM, not avg)')
    print(' 2. fake.detach() during D-step (G frozen)')
    print(' 3. G uses target=1 (real_labels) — wants to fool D')

---
# Section 7: Comparison Drills — RNN vs LSTM Param Count
What's the parameter overhead of LSTM over vanilla RNN at the same hidden size?

In [ ]:
def rnn_params(C, H):
    return 2*H*C + H*H

def lstm_params(D, H):
    # Wx (4H × H+D) + b (4H) + V (D×H) + by (D)
    return 4*H*(H+D) + 4*H + D*H + D

for (C, H) in [(256, 100), (5000, 128), (50000, 512)]:
    rnn = rnn_params(C, H)
    lstm = lstm_params(C, H)  # treating C as D for LSTM
    ratio = lstm / rnn
    print(f'C={C:>6}, H={H:>4}: RNN={rnn:>12,}  LSTM={lstm:>12,}  ratio={ratio:.2f}×')

print('\nLSTM is ~2-4× the params of vanilla RNN at the same hidden size.')
print('Cost: more params. Benefit: solves vanishing gradient via cell state.')

---
# Bonus: Random Practice Generator
Run this cell repeatedly to drill yourself on parameter counting.

In [ ]:
import random

def random_practice():
    qtype = random.choice(['rnn', 'lstm', 'mha', 'conv'])
    if qtype == 'rnn':
        C = random.choice([256, 1000, 5000, 8000, 20000])
        H = random.choice([32, 64, 100, 128, 256])
        ans = 2*H*C + H*H
        return f'Vanilla RNN, vocab C={C}, hidden H={H}. Params (no biases)?', f'2HC + H² = 2·{H}·{C} + {H}² = {ans:,}'
    if qtype == 'lstm':
        D = random.choice([100, 256, 1000])
        H = random.choice([64, 128, 256])
        ans = 4*H*(H+D)
        return f'LSTM gate matrix params (no biases): hidden H={H}, input D={D}?', f'4H(H+D) = 4·{H}·({H}+{D}) = {ans:,}'
    if qtype == 'mha':
        d = random.choice([256, 512, 768, 1024])
        ans = 4*d*d
        return f'MHA params for d_model={d}? (no biases)', f'4d² = 4·{d}² = {ans:,}'
    if qtype == 'conv':
        N, p, f, s = random.choice([(32,1,3,1),(28,0,5,1),(64,2,3,2),(224,3,7,2)])
        ans = (N + 2*p - f) // s + 1
        return f'Conv: input {N}×{N}, kernel {f}×{f}, padding {p}, stride {s}. Output spatial?', f'⌊({N}+{2*p}-{f})/{s}⌋+1 = {ans}'

q, a = random_practice()
print(f'Q: {q}')
print(f'(scroll down or run again to see answer)')
print(f'\n\nA: {a}')

---
# Final 5-Minute Memorization Drill
Read these out loud. They should be reflexive by exam day.

In [ ]:
drills = [
    ('RNN params (no biases)', '2HC + H²'),
    ('LSTM gate params', '4H(H+D)'),
    ('MHA params per layer', '4d²'),
    ('Conv output shape', '⌊(N+2p-f)/s⌋ + 1'),
    ('Transposed conv output', '(N-1)s - 2p + f'),
    ('Attention formula', 'softmax(QK^T / √d_k) V'),
    ('LSTM cell update', 'C_t = f_t·C_{t-1} + i_t·C̃_t'),
    ('GAN D loss', 'L_real + L_fake (SUM)'),
    ("G's loss target", 'real_loss(D(fake), target=1)'),
    ('Per-head dim', 'd_k = d_model / h'),
    ('Numerical gradient', '(L(θ+ε) − L(θ−ε)) / (2ε)'),
    ('Glorot init range', 'U(−1/√n, +1/√n)'),
    ('Forget bias init', '1.0 (sigmoid → 0.73, remember mode)'),
    ('Gradient check threshold', 'rel_err < 1e-3'),
    ('Truncated BPTT length', '4-5 steps'),
]

print(f'{"Concept":<28} | {"Formula / Answer":<40}')
print('-' * 70)
for concept, formula in drills:
    print(f'{concept:<28} | {formula:<40}')
print()
print('🎯 Memorize these. Walk in confident Tuesday.')

---
## ResNet Problems (added topic)

Practice problems on residual networks: parameter counting, shape preservation, gradient flow, and architecture comparisons.

### Problem 6.1 — ResNet basic block parameter count

Count parameters for a basic ResNet block with C=64 channels in/out, two 3×3 convs (no bias, with BatchNorm), identity shortcut.

Expected total: 73,984.

In [ ]:
def basic_block_params(C_in, C_out, stride=1, with_bn=True, bias=False):
    """ResNet basic block: Conv 3x3 -> BN -> ReLU -> Conv 3x3 -> BN -> (+ shortcut) -> ReLU"""
    bias_term = 1 if bias else 0
    conv1 = (3*3*C_in + bias_term) * C_out                  # first conv
    bn1   = 2 * C_out if with_bn else 0                     # BatchNorm
    conv2 = (3*3*C_out + bias_term) * C_out                 # second conv (channels stay)
    bn2   = 2 * C_out if with_bn else 0
    # Shortcut: identity if shape matches, else projection (1x1 conv + BN)
    if stride == 1 and C_in == C_out:
        shortcut = 0                                         # identity, parameter-free
        proj_bn  = 0
    else:
        shortcut = (1*1*C_in + bias_term) * C_out            # 1x1 projection
        proj_bn  = 2 * C_out if with_bn else 0
    total = conv1 + bn1 + conv2 + bn2 + shortcut + proj_bn
    return total, dict(conv1=conv1, bn1=bn1, conv2=conv2, bn2=bn2,
                       shortcut=shortcut, proj_bn=proj_bn)

total, breakdown = basic_block_params(64, 64)
print(f'Basic block (64 -> 64, identity shortcut): {total:,} params')
for k, v in breakdown.items():
    print(f'  {k:10s} = {v:>7,}')
assert total == 73_984, f'expected 73,984, got {total}'
print('\n✓ matches expected 73,984')

### Problem 6.2 — ResNet downsampling block (projection shortcut)

Channel count changes 64 → 128 with stride 2 — needs a projection shortcut. Use the same function with the new arguments and observe that the shortcut now contributes parameters (it's a 1×1 conv).

In [ ]:
total, breakdown = basic_block_params(64, 128, stride=2)
print(f'Downsample block (64 -> 128, projection shortcut): {total:,} params')
for k, v in breakdown.items():
    print(f'  {k:10s} = {v:>7,}')
print(f'\nProjection shortcut adds: {breakdown["shortcut"] + breakdown["proj_bn"]:,} params')
print('  (vs 0 for identity shortcut in the C_in == C_out, stride=1 case)')

### Problem 6.3 — Bottleneck block savings vs basic block

At C=256 channel width, compare a basic block (two 3×3 convs at full width) vs a bottleneck block (1×1 → 3×3 → 1×1, squeezing to C/4 in the middle).

The bottleneck should be ~17× cheaper at the same width — that's why ResNet-50/101/152 use it.

In [ ]:
def bottleneck_block_params(C, with_bn=True):
    """Bottleneck: 1x1 (C -> C/4) -> 3x3 (C/4 -> C/4) -> 1x1 (C/4 -> C). No bias."""
    Cm = C // 4
    conv1 = 1*1*C*Cm                # squeeze
    bn1   = 2 * Cm if with_bn else 0
    conv2 = 3*3*Cm*Cm               # process at low width
    bn2   = 2 * Cm if with_bn else 0
    conv3 = 1*1*Cm*C                # restore
    bn3   = 2 * C  if with_bn else 0
    return conv1 + bn1 + conv2 + bn2 + conv3 + bn3

C = 256
basic, _    = basic_block_params(C, C)
bottleneck  = bottleneck_block_params(C)
print(f'C = {C}')
print(f'  basic block:       {basic:>10,} params')
print(f'  bottleneck block:  {bottleneck:>10,} params')
print(f'  ratio:             {basic / bottleneck:.1f}× cheaper for bottleneck')

### Problem 6.4 — Residual forward pass and gradient flow

Demonstrate two things numerically:
1. When `F(x) ≈ 0`, the residual block's output ≈ input (identity behavior — "do nothing" is easy).
2. When `∂F/∂x ≈ 0`, the residual block's gradient ≈ `∂L/∂y` (the shortcut's `+1` keeps gradient flowing). A plain block would shrink the gradient toward zero.

In [ ]:
# Forward pass
x      = np.array([1.0, 2.0])
F_x    = np.array([0.5, -0.3])      # what the conv layers compute
y_res  = F_x + x                     # residual block output
print('Forward pass:')
print(f'  x        = {x}')
print(f'  F(x)     = {F_x}')
print(f'  y = F+x  = {y_res}')
print()

# When F(x) = 0 (block does nothing)
F_zero = np.zeros(2)
print(f'  If F(x) = 0:  y = F + x = {F_zero + x}  (= x, perfect identity)')
print()

# Gradient flow comparison
dL_dy   = np.array([0.1, -0.2])     # loss gradient at output
dF_dx   = 0.05                       # tiny — block is near-saturating

grad_plain = dL_dy * dF_dx                    # plain block: chain through F only
grad_res   = dL_dy * dF_dx + dL_dy * 1.0      # residual: + shortcut path

print('Gradient flow (assume ∂F/∂x ≈ 0.05):')
print(f'  plain block  ∂L/∂x = {grad_plain}      (shrunk 20× — vanishing!)')
print(f'  residual     ∂L/∂x = {grad_res}     (≈ ∂L/∂y, gradient survives)')
print()
print('  → ResNet\'s identity shortcut keeps gradient alive even when F is near zero.')

### Problem 6.5 — Skip-connection comparison across architectures

Compare the four skip operations covered in this exam. ResNet, LSTM, and Transformer all use additive shortcuts (preserving gradient flow). U-Net concatenates instead (preserving spatial detail).

In [ ]:
comparisons = [
    ('ResNet',      'F(x) + x',                             'add',    'within block, same shape',         'preserve gradient through depth'),
    ('LSTM',        'f_t ⊙ C_{t-1} + i_t ⊙ C̃_t',          'add',    'cell-state through time',           'preserve gradient through time'),
    ('Transformer', 'x + Sublayer(x)',                       'add',    'residual stream around MHA, FFN',  'preserve gradient + enable depth'),
    ('U-Net',       'cat([up, skip], dim=1)',               'concat', 'encoder → decoder, cross-block',   'preserve spatial detail'),
]
print(f'{"Arch":<13}{"Operation":<35}{"Op type":<8}{"Where":<35}{"Purpose"}')
print('-' * 130)
for arch, op, kind, where, purpose in comparisons:
    print(f'{arch:<13}{op:<35}{kind:<8}{where:<35}{purpose}')